## Exercises on Function Approximation and Neural Fitted Q

These paper-and-pencil exercises reinforce Chapter 07: why tabular methods fail in large/continuous spaces, value functions as parameterised approximators $Q(s,a;\theta)$, the mean-squared TD loss, the **semi-gradient** update (treating the bootstrapped target as a constant), mini-batch loss, and the non-stationary-target problem that motivates the next chapter. Notation: $\theta$ are the approximator parameters, $\delta$ the TD error, $\alpha$ the learning rate.

### Exercise 7.1 — The cost of discretising a continuous state space

The Cart-Pole state has $d=4$ continuous variables and $|\mathcal{A}|=2$ actions. Suppose we discretise each state variable into $b$ bins and store a tabular $Q$.

1. How many state cells and how many $Q$-table entries result, as a function of $b$?
2. Evaluate for $b=10$ and $b=20$.
3. Two of the four variables (the velocities) are unbounded — comment on the additional difficulty.

**Step 1 — Count.** With $b$ bins per dimension and $d$ dimensions, the number of discrete states is $b^{d}$; the $Q$-table has one entry per (state, action):

$\displaystyle \#\text{cells}=b^{d}, \qquad \#\text{entries}=|\mathcal{A}|\,b^{d}.$

**Step 2 — Evaluate** ($d=4,\ |\mathcal{A}|=2$):

| $b$ | cells $b^4$ | entries $2b^4$ |
|---|---|---|
| $10$ | $10000$ | $20000$ |
| $20$ | $160000$ | $320000$ |

**Step 3 — Unbounded variables.** The cart and pole *velocities* range over $(-\infty,\infty)$, so any finite binning must first **clip** them to an arbitrary range; states outside the range collapse into edge bins, losing information. Finer binning (larger $b$) reduces this error but makes $b^d$ explode — the **curse of dimensionality**. Most cells are also visited rarely or never, so the table cannot be filled reliably.

**Key concept**

Tabular representation scales as $b^{d}$: exponential in the number of state variables and impossible for continuous states without heavy discretisation. This motivates **function approximation**, which *generalises* across nearby states instead of storing each one.

### Exercise 7.2 — A semi-gradient update with a linear approximator

Use a linear action-value approximator $Q(s,a;\theta)=\theta^\top\phi(s,a)$ with weights $\theta=(0.2,-0.1,0.5)$ and feature vector $\phi(s,a)=(1,0,2)$ for the current pair. A transition gives reward $r=1$, $\gamma=0.9$, and the greedy next-state value is $\max_{a'}Q(s',a';\theta)=2.0$. Learning rate $\alpha=0.1$.

1. Compute the current estimate $Q(s,a;\theta)$.
2. Form the (bootstrapped) TD target and the TD error $\delta$.
3. Perform one **semi-gradient** update $\theta\leftarrow\theta+\alpha\,\delta\,\phi(s,a)$ and recompute $Q(s,a)$.

**Step 1 — Current estimate.** $\ Q(s,a;\theta)=\theta^\top\phi = 0.2(1)+(-0.1)(0)+0.5(2) = 1.2.$

**Step 2 — TD target and error.** The target is treated as a fixed number (it is *detached* from $\theta$):

$\displaystyle \text{target} = r + \gamma\max_{a'}Q(s',a';\theta) = 1 + 0.9(2.0) = 2.8, \qquad \delta = \text{target}-Q = 2.8-1.2 = 1.6.$

**Step 3 — Semi-gradient step.** For a linear model $\nabla_\theta Q(s,a;\theta)=\phi(s,a)$, so

$\displaystyle \theta \leftarrow \theta + \alpha\,\delta\,\phi = (0.2,-0.1,0.5) + 0.1(1.6)(1,0,2) = (0.36, -0.1, 0.82).$

New estimate: $\ Q(s,a;\theta)=\theta^\top\phi = 2$, which has moved from $1.2$ toward the target $2.8$ (by exactly $\alpha\delta\|\phi\|^2 = 0.1(1.6)(5)=0.8$).

**Key concept**

Function approximation replaces a table cell by a *weighted feature sum*; the update nudges the shared weights, so **every state that shares features with $(s,a)$ is affected** — this is generalisation. The gradient of a linear model is just its feature vector.

### Exercise 7.3 — Why "semi-gradient"? (target treated as constant)

The NFQ/TD loss for one transition is $\mathcal{L}(\theta)=\tfrac12\big(y-Q(s,a;\theta)\big)^2$, where the target $y=r+\gamma\max_{a'}Q(s',a';\theta)$ **also depends on $\theta$** (it is produced by the same network). Derive the gradient (i) treating $y$ as a constant (the semi-gradient used in practice), and (ii) differentiating $y$ as well (the full gradient), and explain why the semi-gradient is used.

**Step 1 — Semi-gradient (target detached).** Treating $y$ as a constant label:

$\displaystyle \nabla_\theta \mathcal{L} = -\big(y - Q(s,a;\theta)\big)\,\nabla_\theta Q(s,a;\theta) = -\,\delta\,\nabla_\theta Q(s,a;\theta).$

This gives the update $\theta \leftarrow \theta + \alpha\,\delta\,\nabla_\theta Q(s,a;\theta)$ used in Exercise 7.2.

**Step 2 — Full gradient.** If we also differentiate $y=r+\gamma\max_{a'}Q(s',a';\theta)$ (assume the max is attained at $a'^*$):

$\displaystyle \nabla_\theta \mathcal{L} = -\big(y-Q(s,a;\theta)\big)\Big[\nabla_\theta Q(s,a;\theta) - \gamma\,\nabla_\theta Q(s',a'^*;\theta)\Big].$

The extra term $+\gamma\,\delta\,\nabla_\theta Q(s',a'^*;\theta)$ is what the semi-gradient **drops**.

**Step 3 — Why drop it.** We want $Q(s,a)$ to move *toward* the target, i.e. to treat $y$ as a supervised label — the direction that makes learning behave like regression. Including the second term makes both $Q(s,a)$ and the target chase each other and empirically destabilises learning; it also no longer corresponds to minimising a fixed regression loss (the "target" is not a true constant). Hence RL uses the **semi-gradient**: compute the target, then treat it as a constant when differentiating.

**Key concept**

In supervised learning the target is a fixed label; in bootstrapped RL the target is produced by the very parameters being trained. Pretending it is constant (the **semi-gradient**) is what makes the update well-behaved — and is *the* recurring subtlety of value-based deep RL.

### Exercise 7.4 — Mean-squared loss over a mini-batch

A mini-batch of three transitions has current predictions $Q(s_i,a_i;\theta)=(1.2,\,0.5,\,2.0)$ and (fixed, detached) targets $y_i=(2.8,\,0.5,\,1.0)$. Compute the mean-squared TD loss and state the sign of the update each sample induces.

**Step 1 — Per-sample TD errors** $\delta_i=y_i-Q_i$:

$\displaystyle \delta_1 = 2.8-1.2 = 1.6,\qquad \delta_2 = 0.5-0.5 = 0,\qquad \delta_3 = 1.0-2.0 = -1.0.$

**Step 2 — Mean-squared loss.**

$\displaystyle \mathcal{L} = \frac{1}{3}\sum_{i=1}^{3}\delta_i^2 = \frac{1.6^2 + 0^2 + (-1.0)^2}{3} = \frac{2.56 + 0 + 1.0}{3} = 1.1867.$

**Step 3 — Update directions.** The semi-gradient step moves each prediction toward its target by $\alpha\delta_i$: sample 1 ($\delta_1>0$) is pushed **up**, sample 3 ($\delta_3<0$) is pushed **down**, and sample 2 ($\delta_2=0$) contributes nothing. Because the weights are shared, the net gradient is the average of the per-sample gradients.

**Key concept**

Mini-batch training averages the gradient over several transitions, reducing variance and smoothing the update — but the samples must be reasonably **independent** for this to behave like supervised learning, which sets up the correlation problem addressed next.

### Exercise 7.5 — Chasing a moving target (non-stationarity)

Consider a scalar linear approximator $Q(s;\theta)=\theta\,x(s)$ with a single feature. Two adjacent states share the *same* feature value $x(s)=x(s')=1$ (they look identical to the approximator). A transition $s\to s'$ gives $r=0$, $\gamma=0.9$; start with $\theta=1$ and $\alpha=0.5$.

1. Compute the TD target for this transition, and perform one semi-gradient update of $\theta$.
2. Recompute the target for the **same** transition after the update, and explain what happened.

**Step 1 — Target and update.** Current values: $Q(s)=\theta x(s)=1$, $Q(s')=\theta x(s')=1$.

$\displaystyle \text{target} = r + \gamma Q(s') = 0 + 0.9(1) = 0.9, \qquad \delta = 0.9 - 1 = -0.1.$
$\displaystyle \theta \leftarrow \theta + \alpha\,\delta\,x(s) = 1 + 0.5(-0.1)(1) = 0.95.$

**Step 2 — Recompute the target after the update.** Because $s$ and $s'$ share the feature, updating $\theta$ to fit $s$ also changed $Q(s')$:

$\displaystyle \text{target}' = r + \gamma Q(s';\theta_{\text{new}}) = 0.9\,(0.95)(1) = 0.855.$

The target moved from $0.9$ to $0.855$ — we stepped *toward* a target that then *shifted away*.

**Step 3 — Interpretation.** With a shared/generalising representation, an update aimed at $Q(s)$ perturbs $Q(s')$, so the bootstrapped target is **non-stationary**: it changes every time we train. This "chasing your own tail" is a primary source of instability in naive value-based deep RL.

**Key concept**

Function approximation couples states through shared parameters, making the learning target move as the weights move. Stabilising this — e.g. with a **frozen target network** updated only occasionally — is precisely the fix introduced by DQN in the next chapter.